# Tahap 2f — ICC 271 teks uji, selisih bertanda, verifikasi kriteria konsensus, dan cek `04_review_flag_300`
**CPU biasa, <3 menit.** Tidak ada BERT dan tidak ada notebook lama yang dijalankan ulang.
Sesi 1 = label gold beku (`y_te_v3.npy`, urutan `ho_ids.npy`). Sesi 2 = `06_anotasi_test_271.csv`. Retest 200 = `05_adjudikasi_retest_200.csv`.

In [ ]:
import os, glob, json, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'; DATA4=f'{DRIVE}/JURNAL-DATASET'; OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'; PREP4=f'{OUT4}/prep'
S2=f'{OUT4}/stage2'; os.makedirs(S2,exist_ok=True)
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
def find(name):
    h=sorted(set(glob.glob(f'{DATA4}/**/{name}',recursive=True)+glob.glob(f'{DRIVE}/**/{name}',recursive=True)))
    print(name,'->',h[:3]); return h[0] if h else None


In [ ]:
import numpy as np, pandas as pd
def icc_all(X):
    """X: n x 2 (sesi 1, sesi 2). Mengembalikan ICC(2,1) absolute agreement, ICC(3,1) consistency, dan versi rata-rata (k=2)."""
    X=np.asarray(X,float); n,k=X.shape; gm=X.mean(); rm=X.mean(1); cm=X.mean(0)
    msr=k*((rm-gm)**2).sum()/(n-1); msc=n*((cm-gm)**2).sum()/(k-1)
    mse=((X-rm[:,None]-cm[None,:]+gm)**2).sum()/((n-1)*(k-1))
    return {'ICC2_1_abs_agreement':(msr-mse)/(msr+(k-1)*mse+k*(msc-mse)/n),
            'ICC3_1_consistency':(msr-mse)/(msr+(k-1)*mse),
            'ICC2_k_abs_agreement':(msr-mse)/(msr+(msc-mse)/n),
            'ICC3_k_consistency':(msr-mse)/msr}
def icc_ci(X,B=2000,seed=42):
    rng=np.random.default_rng(seed); X=np.asarray(X,float); n=len(X); base=icc_all(X); bs={k:[] for k in base}
    for _ in range(B):
        r=icc_all(X[rng.integers(0,n,n)])
        for k,v in r.items(): bs[k].append(v)
    return {k:(base[k],np.nanpercentile(bs[k],2.5),np.nanpercentile(bs[k],97.5)) for k in base}

In [ ]:
y_te=np.load(f'{PREP4}/y_te_v3.npy').astype(float); ho_ids=np.load(f'{PREP4}/ho_ids.npy',allow_pickle=True).astype(str)
p06=find('06_anotasi_test_271.csv'); d06=pd.read_csv(p06,encoding='utf-8-sig'); d06['ID_teks']=d06['ID_teks'].astype(str)
S2s=d06.set_index('ID_teks').loc[ho_ids,CATEGORIES].astype(float).values
print('sesi1',y_te.shape,'sesi2',S2s.shape)

In [ ]:
rng=np.random.default_rng(42); rows=[]
for j,c in enumerate(CATEGORIES):
    X=np.c_[y_te[:,j],S2s[:,j]]; res=icc_ci(X); diff=X[:,1]-X[:,0]
    bs=[diff[rng.integers(0,len(diff),len(diff))].mean() for _ in range(2000)]
    from sklearn.metrics import cohen_kappa_score
    row={'Category':c,'n':len(X),'ICC2_1':res['ICC2_1_abs_agreement'][0],'ICC2_1_CI':'[%.2f, %.2f]'%res['ICC2_1_abs_agreement'][1:],
         'ICC3_1':res['ICC3_1_consistency'][0],'ICC3_1_CI':'[%.2f, %.2f]'%res['ICC3_1_consistency'][1:],
         'MeanAbsDiff':np.abs(diff).mean(),'MeanSignedDiff(s2-s1)':diff.mean(),'Signed_CI':'[%.3f, %.3f]'%tuple(np.percentile(bs,[2.5,97.5])),
         'BinaryAgree':((X[:,0]>=.5)==(X[:,1]>=.5)).mean(),'Kappa_bin':cohen_kappa_score((X[:,0]>=.5).astype(int),(X[:,1]>=.5).astype(int))}
    rows.append(row)
t271=pd.DataFrame(rows); print(t271.round(3).T.to_string()); t271.to_csv(f'{S2}/stage2f_icc_271.csv',index=False)
dom=(y_te.argmax(1)==S2s.argmax(1)); print('\nDominant identik antar sesi:',int(dom.sum()),'dari',len(dom),f'({dom.mean()*100:.1f}%)')

In [ ]:
# Verifikasi kriteria 130 konsensus: apakah = teks dengan kategori dominan identik di kedua sesi?
p130=find('test_bersih_130.csv'); t130=pd.read_csv(p130,encoding='utf-8-sig')['ID_teks'].astype(str)
same=set(ho_ids[dom]); print('konsensus 130 file:',len(t130),'| dominan identik:',len(same),'| irisan:',len(set(t130)&same))
print('Binary (4 label >=0.5) identik:',int((((y_te>=.5)==(S2s>=.5)).all(1)).sum()),'| skor identik:',int(np.isclose(y_te,S2s).all(1).sum()))
print('Konsensus = dominan identik persis?', set(t130)==same)

In [ ]:
# Retest 200 : selisih bertanda
p05=find('05_adjudikasi_retest_200.csv'); d05=pd.read_csv(p05,encoding='utf-8-sig'); rows=[]
for c in CATEGORIES:
    X=d05[['orig_'+c,'retest_'+c]].astype(float).values; diff=X[:,1]-X[:,0]
    bs=[diff[rng.integers(0,len(diff),len(diff))].mean() for _ in range(2000)]
    rows.append({'Category':c,'MeanSignedDiff(retest-orig)':diff.mean(),'CI':'[%.3f, %.3f]'%tuple(np.percentile(bs,[2.5,97.5])),'MeanAbsDiff':np.abs(diff).mean()})
r200=pd.DataFrame(rows); print(r200.round(3).to_string(index=False)); r200.to_csv(f'{S2}/stage2f_signed_retest200.csv',index=False)

In [ ]:
# Peran 04_review_flag_300: apakah ID/teksnya ada di data natural (pseudo-label), gold, atau batch lain?
p04=find('04_review_flag_300.csv'); d04=pd.read_csv(p04,encoding='utf-8-sig'); d04['ID_teks']=d04['ID_teks'].astype(str)
norm=lambda s: ' '.join(str(s).lower().split())
n04=set(d04['Segmen_teks'].map(norm)); i04=set(d04['ID_teks'])
for name in ['anotasi_natural_2000.csv','data_ahli_FIXED.csv','gold_batch3_577.csv','gold_batch4_600.csv','gold_batch5_600.csv']:
    p=find(name)
    if not p: continue
    d=pd.read_csv(p,encoding='utf-8-sig'); idc=[c for c in d.columns if c.lower().startswith('id')][0]; tc=[c for c in d.columns if 'segmen' in c.lower() or 'teks' in c.lower() and c!=idc][0]
    print(f'{name:28s} n={len(d):5d} | irisan ID: {len(i04&set(d[idc].astype(str))):3d} | irisan teks: {len(n04&set(d[tc].map(norm))):3d}')
print('\nKolom kategori_label_otomatis:',d04['kategori_label_otomatis'].value_counts().to_dict())

In [ ]:
print(open(f'{S2}/stage2f_icc_271.csv').read()[:1500])